<a href="https://colab.research.google.com/github/josh777-ops/Fly-Rank-AI/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josh777-ops/Fly-Rank-AI/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 code cell
print("Research question: Can same-day behavioral signals (impressions, engaged")
print("sessions, traffic source, AI-referral sessions, scroll activity) predict a")
print("content item's average Google search position for a client not seen during training?")
print()
print("Decision this supports: whether a content team should trust a predictive model")
print("to prioritize which pages to review for ranking fixes, or rely on a simpler,")
print("validated rule instead.")


Research question: Can same-day behavioral signals (impressions, engaged
sessions, traffic source, AI-referral sessions, scroll activity) predict a
content item's average Google search position for a client not seen during training?

Decision this supports: whether a content team should trust a predictive model
to prioritize which pages to review for ranking fixes, or rely on a simpler,
validated rule instead.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2 code cell
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

# Grain check
grain = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_grain
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(grain)

# Availability
avail = con.sql(f"""
    SELECT gsc_data_available, COUNT(*) AS n
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY gsc_data_available
""").df()
print(avail)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  distinct_grain
0     9841378         9841378


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   gsc_data_available        n
0               False  6230317
1                True  3611061


Release: FlyRank/internship-warehouse, table fact_content_daily_performance, partition month=2026-03. Grain verified: total rows == distinct (report_date, client_hash_id, content_hash_id), confirming one row = one content item's daily performance for one client. Date window: 2026-03-01 to 2026-03-31, a mid-panel month (not the sealed final month). Excluded: client_hash_id/content_hash_id used only for grouping, never as features; gsc_sum_position/gsc_clicks excluded as label-derived (demonstrated in Methodology); zero-impression rows dropped before modeling since position is undefined without impressions.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 code cell
# Signal checks
sig1 = con.sql(f"""
    SELECT CASE WHEN gsc_avg_position <= 3 THEN '1-3' WHEN gsc_avg_position <= 10 THEN '4-10'
                WHEN gsc_avg_position <= 20 THEN '11-20' ELSE '20+' END AS position_bucket,
           AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) AS avg_ctr, COUNT(*) AS n
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY position_bucket ORDER BY position_bucket
""").df()
print("Signal 1 (CTR vs position):")
print(sig1)

# Leakage audit
df = con.sql(f"""
    SELECT gsc_avg_position, gsc_impressions, ga4_engaged_sessions, sessions_organic,
           sessions_ai, scroll_events, gsc_sum_position
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df().dropna()

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X_leak = df[['gsc_impressions','ga4_engaged_sessions','sessions_organic','sessions_ai','scroll_events','gsc_sum_position']]
X_honest = df[['gsc_impressions','ga4_engaged_sessions','sessions_organic','sessions_ai','scroll_events']]
y = df['gsc_avg_position']

Xtr, Xte, ytr, yte = train_test_split(X_leak, y, test_size=0.2, random_state=42)
r2_leak = r2_score(yte, LinearRegression().fit(Xtr, ytr).predict(Xte))
Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.2, random_state=42)
r2_honest = r2_score(yte, LinearRegression().fit(Xtr, ytr).predict(Xte))
print(f"\nLeakage audit — WITH leak R²: {r2_leak:.4f}, WITHOUT leak R²: {r2_honest:.4f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1 (CTR vs position):
  position_bucket   avg_ctr        n
0             1-3  0.004756   727362
1           11-20  0.002770   519223
2             20+  0.001289   908354
3            4-10  0.003473  1456122


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Leakage audit — WITH leak R²: 0.0624, WITHOUT leak R²: 0.0033


Label: gsc_avg_position. Features (5): gsc_impressions, ga4_engaged_sessions, sessions_organic, sessions_ai, scroll_events. Baseline rule: VISIBLE_BUT_POOR_RANK when impressions > 0 and position > 10, scored as impressions × (position − 10). Validation: GroupShuffleSplit by client_hash_id, zero client overlap verified. Leakage audit confirms gsc_sum_position is a real leak source (~19× R² inflation), correctly excluded from the final feature set.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 4 code cell
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

groups = df['client_hash_id'] if 'client_hash_id' in df.columns else None
# re-pull with client_hash_id for grouping
df_g = con.sql(f"""
    SELECT client_hash_id, gsc_avg_position, gsc_impressions, ga4_engaged_sessions,
           sessions_organic, sessions_ai, scroll_events
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df().dropna()

X = df_g[['gsc_impressions','ga4_engaged_sessions','sessions_organic','sessions_ai','scroll_events']]
y = df_g['gsc_avg_position']
groups = df_g['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
Xtr, Xte, ytr, yte = X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]

lr = LinearRegression().fit(Xtr, ytr)
rf = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1).fit(Xtr, ytr)

import pandas as pd
results = pd.DataFrame({
    'Model': ['Linear Regression (baseline)', 'Random Forest'],
    'R2_grouped_split': [r2_score(yte, lr.predict(Xte)), r2_score(yte, rf.predict(Xte))],
    'MAE': [mean_absolute_error(yte, lr.predict(Xte)), mean_absolute_error(yte, rf.predict(Xte))]
})
print(results)

                          Model  R2_grouped_split        MAE
0  Linear Regression (baseline)         -0.013452  11.506883
1                 Random Forest         -0.007388  11.428573


Both models score negative R² under the honest, client-grouped split — neither beats predicting the mean for an unseen client. This is a real regression from an earlier ungrouped-split result that looked positive (R² = 0.0033), revealing that number was inflated by client-identity leakage.

## 5. Limitations

*What this work cannot claim.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 5 code cell
before_dropna = con.sql(f"""
    SELECT COUNT(*) AS n
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df()

print("GSC availability breakdown:")
print(avail)
print()
print("Rows before dropna (zero-impression exclusion):", before_dropna['n'][0])
print("Rows after dropna:", len(df_g))
print("Rows removed:", before_dropna['n'][0] - len(df_g))
print("Percent removed:", round((before_dropna['n'][0] - len(df_g)) / before_dropna['n'][0] * 100, 1), "%")


GSC availability breakdown:
   gsc_data_available        n
0               False  6230317
1                True  3611061

Rows before dropna (zero-impression exclusion): 3611061
Rows after dropna: 2082695
Rows removed: 1528366
Percent removed: 42.3 %


Unbalanced panel (36.7% GSC availability, not random). Zero-impression exclusion removes rows with an undefined target, skewing toward already-visible content. Observational, not causal. Negative R² is specific to this feature set/window/lane, not a claim about all FlyRank data.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 6 code cell — corrected, self-contained pull
queue_df = con.sql(f"""
    SELECT client_hash_id, content_hash_id, gsc_avg_position, gsc_impressions,
           ga4_engaged_sessions, sessions_organic, sessions_ai, scroll_events
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df().dropna()

queue_df['reason_code'] = queue_df.apply(
    lambda r: 'NO_VISIBILITY' if r['gsc_impressions']==0
    else ('RANKS_WELL' if r['gsc_avg_position']<=10 else 'VISIBLE_BUT_POOR_RANK'),
    axis=1
)
queue_df['score'] = queue_df.apply(
    lambda r: r['gsc_impressions']*(r['gsc_avg_position']-10) if r['reason_code']=='VISIBLE_BUT_POOR_RANK' else 0,
    axis=1
)

threshold = queue_df.loc[queue_df['score'] > 0, 'score'].quantile(0.95)
queue_df['action'] = queue_df['score'].apply(lambda s: 'REVIEW_FOR_RANKING_FIX' if s >= threshold else 'MONITOR')

queue = (queue_df[queue_df['action']=='REVIEW_FOR_RANKING_FIX']
    .groupby(['client_hash_id','content_hash_id'])
    .agg(avg_score=('score','mean'), days_flagged=('score','count'))
    .reset_index())

print("Unique content items flagged:", len(queue))
print(queue.sort_values('avg_score', ascending=False).head(5))



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unique content items flagged: 3925
               client_hash_id           content_hash_id      avg_score  \
1288  client_23a62021009f63c4  content_36e53e9c707674fc  143028.000000   
1740  client_23a62021009f63c4  content_6530fa9d297c46eb  138552.000000   
1895  client_23a62021009f63c4  content_73aa61dcedebbf30   94038.258065   
1908  client_23a62021009f63c4  content_74de5f247659e956   93498.000000   
1583  client_23a62021009f63c4  content_559cdd76da9306de   84493.870968   

      days_flagged  
1288            31  
1740             5  
1895            31  
1908             1  
1583            31  


Ranked recommendations, reused from the Week 7 action playbook: (1) prioritize items flagged across most/all of the month over single-day spikes; (2) treat the queue as a human-review shortlist, not an automated action list; (3) do not deploy the regression model — it did not beat baseline; (4) re-verify the CTR-vs-position signal quarterly; (5) never use the queue alone to justify budget/headcount decisions.

Note: this figure (3,925) differs from an earlier, separate run of this same rule (6,036, in w07_action_playbook.ipynb). The difference traces to this notebook applying .dropna() before scoring — consistent with the rest of this capstone's pipeline — while the earlier notebook scored against the full availability-filtered set without that step. This capstone notebook's queue (3,925) is the one the deployed paper's numbers are drawn from, since it's built on the same consistently-filtered data used throughout this notebook.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 7 code cell
import os, json
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

results.to_json('work/outputs/model_vs_baseline.json', orient='records', indent=2)
sig1.to_json('work/outputs/signal1_ctr_vs_position.json', orient='records', indent=2)

capstone_metrics = {
    "grain_verified": True,
    "total_rows": int(grain['total_rows'][0]),
    "gsc_availability_pct": 36.7,
    "leakage_r2_with": float(r2_leak),
    "leakage_r2_without": float(r2_honest),
    "final_r2_baseline": float(results['R2_grouped_split'][0]),
    "final_r2_model": float(results['R2_grouped_split'][1]),
}
with open('work/outputs/capstone_metrics.json', 'w') as f:
    json.dump(capstone_metrics, f, indent=2)

print("Artifacts exported — these are the exact numbers the deployed paper embeds.")

Artifacts exported — these are the exact numbers the deployed paper embeds.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.